In [6]:
!apt-get install swig -y -q
!pip install "gymnasium[box2d]" stable-baselines3[extra] -q

Reading package lists...
Building dependency tree...
Reading state information...
swig is already the newest version (4.0.2-1ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [10]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env

In [9]:
env_id = "LunarLander-v3"

env = make_vec_env(env_id, n_envs=8)

In [11]:
model = PPO(
    policy="MlpPolicy",
    env=env,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.01,
    verbose=1,
    tensorboard_log="./ppo_lunarlander_tensorboard/"
)


model.learn(total_timesteps=1_000_000)

model.save("ppo_lunarlander")

Using cpu device
Logging to ./ppo_lunarlander_tensorboard/PPO_2
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 95.1     |
|    ep_rew_mean     | -189     |
| time/              |          |
|    fps             | 3059     |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 8192     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 94.2        |
|    ep_rew_mean          | -171        |
| time/                   |             |
|    fps                  | 2176        |
|    iterations           | 2           |
|    time_elapsed         | 7           |
|    total_timesteps      | 16384       |
| train/                  |             |
|    approx_kl            | 0.006692568 |
|    clip_fraction        | 0.00555     |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.38       |
|    exp

In [12]:
eval_env = gym.make(env_id)
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=20)
print(f"Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Mean reward: 235.31 +/- 66.03


In [13]:
!pip install imageio imageio-ffmpeg -q

import imageio

env = gym.make(env_id, render_mode="rgb_array")
obs, _ = env.reset()
frames = []

for _ in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    frames.append(env.render())
    if terminated or truncated:
        obs, _ = env.reset()

imageio.mimsave("lunarlander_agent.mp4", frames, fps=30)
env.close()

In [14]:
from IPython.display import Video
Video("lunarlander_agent.mp4", embed=True)